# Forecast Rationale Generator

**Project:** Financial Planning & Analysis Intelligence Platform

**Notebook:** `13-forecast-rationale-generator.ipynb`

In [1]:
# ==========================================
# Notebook 13
# Forecast Rationale Engine
# ==========================================

import pandas as pd
import numpy as np

import faiss

from sentence_transformers import SentenceTransformer

from xgboost import XGBRegressor

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
forecast_df = pd.read_csv("../data/multimodal_forecasting_dataset.csv")

financial_corpus_df = pd.read_csv("../data/financial_rag_corpus.csv")

forecast_df.head()

,gross_margin_pct,operating_income_million,net_income_million,eps,weighted_sentiment,guidance_score,positive_guidance_score,total_risk_score,normalized_risk_score,macro_risk_score,revenue_million
0,58,22,16,1.20,0.956167,1,1,2,1.0,0.583904,120
1,60,25,18,1.35,0.957692,1,2,1,0.5,0.624035,128
2,61,28,21,1.52,0.951876,2,0,1,0.5,0.565232,138
3,63,32,24,1.72,0.956653,1,1,0,0.0,0.499716,150
4,62,34,26,1.85,0.922371,1,0,0,0.0,0.381064,158


In [3]:
financial_corpus_df

,ticker,quarter,revenue_million,gross_margin_pct,operating_income_million,net_income_million,eps,earnings_call,risk_factors,mda_section,...,weighted_sentiment,guidance_score,positive_guidance_score,total_risk_score,normalized_risk_score,financial_document,best_macro_event,similarity_score,macro_risk_score,rag_document
0,ABC,2023-Q1,120,58,22,16,1.20,\n Demand remained strong across enterp...,\n Inflation remains a concern.\n ...,\n Management believes demand trends re...,...,0.956167,1,1,2,1.0,\n Demand remained strong across enterp...,High Inflation Environment,0.583904,0.583904,\n Demand remained strong across enterp...
1,ABC,2023-Q2,128,60,25,18,1.35,\n Enterprise adoption accelerated.\n ...,\n Competitive pressure exists.\n ...,\n New product launches contributed pos...,...,0.957692,1,2,1,0.5,\n Enterprise adoption accelerated.\n ...,Currency Volatility Event,0.624035,0.624035,\n Enterprise adoption accelerated.\n ...
2,ABC,2023-Q3,138,61,28,21,1.52,\n Customer demand exceeded expectation...,\n Geopolitical uncertainty remains.\n ...,\n Strong customer growth across region...,...,0.951876,2,0,1,0.5,\n Customer demand exceeded expectation...,Regulatory Pressure Cycle,0.565232,0.565232,\n Customer demand exceeded expectation...
3,ABC,2023-Q4,150,63,32,24,1.72,\n Record quarter performance.\n ...,\n Macroeconomic slowdown remains possi...,\n Revenue growth exceeded internal exp...,...,0.956653,1,1,0,0.0,\n Record quarter performance.\n ...,Interest Rate Shock,0.499716,0.499716,\n Record quarter performance.\n ...
4,ABC,2024-Q1,158,62,34,26,1.85,\n Strong start to the fiscal year driv...,\n Increased cyber security threats req...,\n Gross margin slightly contracted due...,...,0.922371,1,0,0,0.0,\n Strong start to the fiscal year driv...,COVID Supply Chain Crisis,0.381064,0.381064,\n Strong start to the fiscal year driv...
5,ABC,2024-Q2,167,64,38,29,2.05,\n New AI-driven product modules saw re...,\n Intense competition in the mid-marke...,\n Expansion of gross margin driven by ...,...,0.921203,2,0,2,1.0,\n New AI-driven product modules saw re...,High Inflation Environment,0.541657,0.541657,\n New AI-driven product modules saw re...
6,ABC,2024-Q3,174,64,41,31,2.18,\n European expansion is yielding solid...,\n Evolving global data privacy laws co...,\n Operating margin expanded due to sca...,...,0.919702,1,0,1,0.5,\n European expansion is yielding solid...,Regulatory Pressure Cycle,0.312966,0.312966,\n European expansion is yielding solid...
7,ABC,2024-Q4,190,65,46,35,2.45,"\n An exceptional finish to 2024, cross...",\n Potential shifts in corporate tax ra...,\n Record annual performance driven by ...,...,0.952842,1,2,0,0.0,"\n An exceptional finish to 2024, cross...",COVID Supply Chain Crisis,0.421793,0.421793,"\n An exceptional finish to 2024, cross..."
8,ABC,2025-Q1,198,64,48,36,2.52,\n We carried the strong closing moment...,\n Geopolitical trade restrictions coul...,\n Growth continues at a stable pace de...,...,-0.740657,2,2,0,0.0,\n We carried the strong closing moment...,COVID Supply Chain Crisis,0.584518,0.584518,\n We carried the strong closing moment...
9,ABC,2025-Q2,210,66,53,40,2.80,\n Our SaaS migration is officially com...,\n Slowing down of tech spending in the...,\n Gross margins hit a record high due ...,...,0.953374,1,0,0,0.0,\n Our SaaS migration is officially com...,Interest Rate Shock,0.514108,0.514108,\n Our SaaS migration is officially com...


In [4]:
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [5]:
financial_corpus_df["rag_document"] = (
    financial_corpus_df["earnings_call"].fillna("")
    + " "
    + financial_corpus_df["risk_factors"].fillna("")
    + " "
    + financial_corpus_df["mda_section"].fillna("")
)

In [6]:
document_embeddings = embedding_model.encode(
    financial_corpus_df["rag_document"].tolist(), show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
dimension = document_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(document_embeddings).astype("float32"))

In [8]:
feature_columns = [
    "gross_margin_pct",
    "operating_income_million",
    "net_income_million",
    "eps",
    "weighted_sentiment",
    "guidance_score",
    "positive_guidance_score",
    "total_risk_score",
    "normalized_risk_score",
    "macro_risk_score",
]

X = forecast_df[feature_columns]

y = forecast_df["revenue_million"]

forecast_model = XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42
)

forecast_model.fit(X, y)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [9]:
importance_df = pd.DataFrame(
    {"feature": feature_columns, "importance": forecast_model.feature_importances_}
)

importance_df = importance_df.sort_values(by="importance", ascending=False)

importance_df

,feature,importance
1,operating_income_million,0.942925
0,gross_margin_pct,0.057075
2,net_income_million,0.000000
3,eps,0.000000
4,weighted_sentiment,0.000000
5,guidance_score,0.000000
6,positive_guidance_score,0.000000
7,total_risk_score,0.000000
8,normalized_risk_score,0.000000
9,macro_risk_score,0.000000


In [10]:
def retrieve_evidence(query, top_k=3):

    query_embedding = embedding_model.encode(query)

    distances, indices = index.search(
        np.array([query_embedding]).astype("float32"), top_k
    )

    evidence = []

    for idx in indices[0]:

        evidence.append(financial_corpus_df.iloc[idx]["rag_document"])

    return evidence

In [11]:
retrieve_evidence("strong customer demand")

['\n        Demand remained strong across enterprise customers.\n        We expect continued growth next quarter.\n        Supply chain constraints are gradually easing.\n         \n        Inflation remains a concern.\n        Global logistics costs remain elevated.\n         \n        Management believes demand trends remain healthy.\n        Customer retention continues to improve.\n        ',
 '\n        Customer demand exceeded expectations.\n        Supply chain bottlenecks largely resolved.\n        International expansion continues.\n         \n        Geopolitical uncertainty remains.\n        Regulatory changes may affect operations.\n         \n        Strong customer growth across regions.\n        Profitability improved significantly.\n        ',
 '\n        Record quarter performance.\n        Management expects continued growth.\n        Demand remains exceptionally strong.\n         \n        Macroeconomic slowdown remains possible.\n        Talent acquisition costs con

In [12]:
future_quarter = pd.DataFrame(
    {
        "gross_margin_pct": [65],
        "operating_income_million": [36],
        "net_income_million": [28],
        "eps": [2.0],
        "weighted_sentiment": [0.92],
        "guidance_score": [4],
        "positive_guidance_score": [3],
        "total_risk_score": [1],
        "normalized_risk_score": [0.25],
        "macro_risk_score": [0.30],
    }
)

In [13]:
predicted_revenue = forecast_model.predict(future_quarter)[0]

predicted_revenue

167.00244

In [14]:
top_drivers = importance_df.head(5)

top_drivers

,feature,importance
1,operating_income_million,0.942925
0,gross_margin_pct,0.057075
2,net_income_million,0.000000
3,eps,0.000000
4,weighted_sentiment,0.000000


In [15]:
evidence_results = []

for feature in top_drivers["feature"]:

    evidence = retrieve_evidence(feature)

    evidence_results.append({"feature": feature, "evidence": evidence[0]})

In [19]:
evidence_df = pd.DataFrame(evidence_results)

evidence_df

,feature,evidence
0,operating_income_million,\n Record quarter performance.\n ...
1,gross_margin_pct,\n We carried the strong closing moment...
2,net_income_million,\n Record quarter performance.\n ...
3,eps,"\n A monumental year for ABC, closing o..."
4,weighted_sentiment,\n New AI-driven product modules saw re...


In [17]:
rationale = f"""

Predicted Revenue:
${predicted_revenue:.2f} Million

Key Forecast Drivers:

"""

for _, row in top_drivers.iterrows():

    rationale += f"""

- {row['feature']}
  Importance: {row['importance']:.4f}

"""

In [20]:
print(rationale)



Predicted Revenue:
$167.00 Million

Key Forecast Drivers:



- operating_income_million
  Importance: 0.9429



- gross_margin_pct
  Importance: 0.0571



- net_income_million
  Importance: 0.0000



- eps
  Importance: 0.0000



- weighted_sentiment
  Importance: 0.0000




In [21]:
summary = f"""

EXECUTIVE FORECAST SUMMARY

Forecast Revenue:
${predicted_revenue:.2f} Million

Positive Signals:

• Strong management sentiment

• Positive forward guidance

• Healthy profitability

Risk Signals:

• Existing macroeconomic uncertainty

• Potential regulatory pressure

Overall Assessment:

Revenue outlook remains positive
with manageable risk exposure.

"""

In [22]:
print(summary)



EXECUTIVE FORECAST SUMMARY

Forecast Revenue:
$167.00 Million

Positive Signals:

• Strong management sentiment

• Positive forward guidance

• Healthy profitability

Risk Signals:

• Existing macroeconomic uncertainty

• Potential regulatory pressure

Overall Assessment:

Revenue outlook remains positive
with manageable risk exposure.




In [23]:
audit_report = {
    "predicted_revenue": float(predicted_revenue),
    "top_features": top_drivers["feature"].tolist(),
    "feature_importance": top_drivers["importance"].tolist(),
    "supporting_evidence": evidence_df["evidence"].tolist(),
}

In [24]:
audit_report

{'predicted_revenue': 167.00244140625,
 'top_features': ['operating_income_million',
  'gross_margin_pct',
  'net_income_million',
  'eps',
  'weighted_sentiment'],
 'feature_importance': [0.9429254531860352,
  0.057074546813964844,
  0.0,
  0.0,
  0.0],
 'supporting_evidence': ['\n        Record quarter performance.\n        Management expects continued growth.\n        Demand remains exceptionally strong.\n         \n        Macroeconomic slowdown remains possible.\n        Talent acquisition costs continue rising.\n         \n        Revenue growth exceeded internal expectations.\n        Operating leverage improved.\n        ',
  '\n        We carried the strong closing momentum of last year into Q1.\n        Professional services revenue saw an unexpected upside.\n        Implementation timelines for major cloud migrations have shortened.\n         \n        Geopolitical trade restrictions could indirectly affect our hardware partners.\n        Higher energy and cloud data center 

In [27]:
rationale_df = pd.DataFrame(
    {"feature": top_drivers["feature"], "importance": top_drivers["importance"]}
)

rationale_df

,feature,importance
1,operating_income_million,0.942925
0,gross_margin_pct,0.057075
2,net_income_million,0.000000
3,eps,0.000000
4,weighted_sentiment,0.000000


In [28]:
rationale_df.to_csv("../data/forecast_rationale.csv", index=False)

In [29]:
pd.DataFrame(audit_report).to_csv("../data/forecast_audit_report.csv", index=False)

In [30]:
with open("../data/executive_forecast_summary.txt", "w", encoding="utf-8") as file:

    file.write(summary)

In [31]:
pd.read_csv("../data/forecast_rationale.csv")

,feature,importance
0,operating_income_million,0.942925
1,gross_margin_pct,0.057075
2,net_income_million,0.000000
3,eps,0.000000
4,weighted_sentiment,0.000000
